# SAC arrival_v2 — AsymCritic ablation on single_cross_s0 (seed=42, 1M, pure ablation)

**Pre-context**：commit `01b78ad` 已闭环 [`docs/arrival_v2_experiment_report.md`](../docs/arrival_v2_experiment_report.md) §7.6（s0 sensor envelope）。结果：

| §7.6 cell | benchmark | sensor | algo | final | OOB | gate |
|---|---|---|---|---:|---:|---|
| §7.6.1 | `tandem_u15_upstream_tgt15` | s0_k4 | vanilla | 1.000 | 0.000 | PASS |
| §7.6.2 | `sbs_u15_upstream_tgt15` | s0_k4 | vanilla | 1.000 | 0.000 | PASS |
| §7.6.3 | `single_u15_upstream_tgt15` | s0_k4 | vanilla | 1.000 | 0.000 | PASS |
| **§7.6.4** | **`single_u15_cross_tgt15`** | **s0_k4** | **vanilla** | **0.100** | **0.667** | **FAIL** |

→ 8 cell envelope（4 topology × 2 sensor）中只有 `single_cross_s0` 一个 cell 没 PASS，且 final 比 s1 (0.900) 低 80pp。

**本 notebook 任务（pure AsymCritic ablation）**：把 §7.6.4 的 vanilla SAC **唯一变量**换成开启 AsymCritic（`--use-asymmetric-critic`），其它**全部不动**：

| 维度 | §7.6.4 vanilla baseline | 本 notebook |
|---|---|---|
| algorithm | vanilla SAC | **vanilla SAC + AsymCritic** ← 唯一变量 |
| `--use-layernorm` | off | **off**（保持 vanilla 网络） |
| `--updates-per-step` | 1 | **1**（保持 vanilla UTD） |
| sensor layout | s0_k4 (10-D) | s0_k4 (10-D) |
| reward | arrival_v2 | arrival_v2 |
| flow U / target | 1.5 / 1.5 | 1.5 / 1.5 |
| seed | 42 | 42 |
| total_steps | 1M | 1M |
| num_envs | 6 | 6 |
| benchmark | `single_u15_cross_tgt15` | 同 |

跑完后 critic 看到的额外信息是 `privileged_obs = [u_eq, v_eq]`（body-frame hull-integral 等效流），actor 仍只有 s0 单点采样，与 deployment 一致。**任何差异只能归因于 AsymCritic 的训练信号差异**。

**为什么是 pure ablation（B），不是 sac_asym_lnutd 组合（A）？** 见 chat 决策：组合改动 (AsymCritic + LayerNorm + UTD=4) 一起跑通无法归因；先用最小变量（只加 AsymCritic）测 [`docs/SAC_improvements_survey.md`](../docs/SAC_improvements_survey.md) §10.2 P1#5 留口 = "privileged hull-integral flow 对本任务理论收益最高"。若闭合 → 论文直接写 AsymCritic 单独贡献；若不闭合 → 增量跑 A 组合，再看是否其它改进配合后才有用。

**Gate**（与 §7 / §7.6 同口径）：
- `final_success_rate ≥ 0.85`
- `last100k_mean ≥ 0.9 × peak`
- `final_oob_rate ≤ 0.10`
- `include_episode_context_obs == True`
- `timeout_bootstrap_semantics == 'terminal'`

**Single seed (=42)**：与 §7.6 同步。**Exploratory**，不是 thesis-grade。判读规则：
- 若 PASS（final ≥ 0.85, OOB ≤ 0.10）→ AsymCritic 单独闭合了 80pp gap。论点："privileged hull-integral flow 显著缓解 cross-flow 几何下 s0 信息瓶颈"。下一步 multi-seed 复现。
- 若 partial（final ∈ [0.4, 0.85)）→ AsymCritic 有效但不充分。下一步走 A 路径（叠加 LN + UTD=4 看是否补齐）。
- 若 fail（final < 0.4）→ AsymCritic 对本 failure mode 无显著作用。论点："single_cross_s0 的 collapse 源不在 critic estimation 误差，可能在 reward shaping 或探索"。

**输出根（与 §7.6 baseline 平行；`sac_vanilla` → `sac_asym`，互不覆盖）**：
- `experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_asym/s0_k4/seed_42/`

**总预算**：~2.5h L4（1 Colab Pro+ session）。

**风格**：训练用 `!python -u -m scripts.train_sac` 直跑（实时 stdout），与 §7 / §7.6 系列 notebook 一致。


## 0. GPU sanity


In [1]:
!nvidia-smi | head -10


Sun May 17 15:46:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   35C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |


## 1. Mount Drive + cwd


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

REPO_DIR = '/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5'
%cd $REPO_DIR


Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5


## 2. Config — single phase（pure AsymCritic ablation, 与 §7.6.4 vanilla 仅差一个 flag）


In [3]:
import json
import os
from pathlib import Path

import pandas as pd

# ==== SAC / env config (与 §7.6.4 baseline 严格一致，除算法外) ====
OBJECTIVE = 'arrival_v2'
PROBE_LAYOUT = 's0'
HISTORY_LENGTH = 4
TARGET_SPEED = 1.5
SEED = 42

RANDOM_STEPS = 5_000
UPDATE_AFTER = 5_000
BATCH_SIZE = 256
HIDDEN_DIM = 256
NUM_ENVS = 6
EVAL_EVERY = 25_000
EVAL_EPISODES = 30
CHECKPOINT_EVERY = 100_000
DEVICE = 'cuda'

# AsymCritic 是 pure ablation 唯一新增项
USE_ASYMMETRIC_CRITIC = True
# 显式拒绝其它改进项 — 与 vanilla baseline 仅差 AsymCritic 一项
USE_LAYERNORM = False
UPDATES_PER_STEP = 1
DROPOUT_RATE = 0.0

PASS_FINAL_SUCCESS = 0.85
PASS_LAST100_RATIO = 0.90
PASS_OOB_RATE = 0.10

# Flow file（与 §7.1 / §7.6.4 严格一致）
SINGLE_FLOW = 'wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy'

# ==== Single phase — single_cross s0 AsymCritic (X_*) ====
X_BENCHMARK_KEY = 'single_u15_cross_tgt15'
X_TASK_GEOMETRY = 'cross_stream'
X_FLOW_PATH = SINGLE_FLOW
X_TOTAL_STEPS = 1_000_000
X_RUN_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_asym/s0_k4/seed_42')
X_CKPT_ROOT = Path('checkpoints/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_asym/s0_k4/seed_42')
X_MANIFEST_PATH = Path(f'benchmarks/{X_BENCHMARK_KEY}.json')

# §7.6.4 vanilla baseline (s0_k4) — 用于事后对比
X_VANILLA_BASELINE_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_42')
# §7.1 vanilla baseline (s1_k4) — reference upper bound
X_S1_BASELINE_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42')

os.environ['PYTHONUNBUFFERED'] = '1'

print(f'PROBE_LAYOUT          = {PROBE_LAYOUT}')
print(f'OBJECTIVE             = {OBJECTIVE}')
print(f'HISTORY               = k={HISTORY_LENGTH}')
print(f'SEED                  = {SEED}')
print(f'NUM_ENVS              = {NUM_ENVS}')
print(f'USE_ASYMMETRIC_CRITIC = {USE_ASYMMETRIC_CRITIC}  ← 唯一与 §7.6.4 vanilla 的差异')
print(f'USE_LAYERNORM         = {USE_LAYERNORM}         (off — 保持 vanilla 网络)')
print(f'UPDATES_PER_STEP      = {UPDATES_PER_STEP}      (=1, 保持 vanilla UTD)')
print(f'DROPOUT_RATE          = {DROPOUT_RATE}          (off)')
print()
print(f'benchmark             : {X_BENCHMARK_KEY}')
print(f'geometry              : {X_TASK_GEOMETRY}')
print(f'total_steps           : {X_TOTAL_STEPS:,}')
print(f'run_root              : {X_RUN_ROOT}')
print(f'vanilla baseline ref  : {X_VANILLA_BASELINE_ROOT}')
print(f's1 baseline ref       : {X_S1_BASELINE_ROOT}')


PROBE_LAYOUT          = s0
OBJECTIVE             = arrival_v2
HISTORY               = k=4
SEED                  = 42
NUM_ENVS              = 6
USE_ASYMMETRIC_CRITIC = True  ← 唯一与 §7.6.4 vanilla 的差异
USE_LAYERNORM         = False         (off — 保持 vanilla 网络)
UPDATES_PER_STEP      = 1      (=1, 保持 vanilla UTD)
DROPOUT_RATE          = 0.0          (off)

benchmark             : single_u15_cross_tgt15
geometry              : cross_stream
total_steps           : 1,000,000
run_root              : experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_asym/s0_k4/seed_42
vanilla baseline ref  : experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_42
s1 baseline ref       : experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42


## 3. Preflight — flow / arrival_v2 candidate gate / reward unit tests / manifest / AsymCritic wiring sanity


In [4]:
# Flow file
fp = Path(X_FLOW_PATH)
if not fp.exists():
    raise FileNotFoundError(f'missing flow file: {fp}')
print(f'[OK] flow file: {fp}  ({fp.stat().st_size / 1e6:.1f} MB)')


[OK] flow file: wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy  (230.4 MB)


In [5]:
!python -u -m scripts.validate_arrival_v2_candidate



[undiscounted]
fast_success       144.473
slow_success       142.023
unsafe_success      91.048
timeout_near      -149.800
timeout_far       -229.800
late_oob          -274.500
fast_oob          -280.325
mid_oob           -319.450

[discounted_gamma_0.995]
fast_success        88.578
slow_success        54.683
unsafe_success      34.857
timeout_near       -39.480
timeout_far        -58.269
late_oob          -103.800
mid_oob           -190.006
fast_oob          -212.555

[discounted_shortcut] safe=69.330 risky=62.361

PASS: arrival_v2 candidate pre-integration gates passed.


In [6]:
!python -u -m pytest tests/test_reward_objective.py -q


...............                                                          [100%]
15 passed in 21.09s


In [7]:
if not X_MANIFEST_PATH.exists():
    !python -u -m scripts.generate_standard_benchmarks --benchmarks {X_BENCHMARK_KEY} --episodes {EVAL_EPISODES}
if not X_MANIFEST_PATH.exists():
    raise FileNotFoundError(f'manifest not generated: {X_MANIFEST_PATH}')
print(f'[OK] manifest ready: {X_MANIFEST_PATH}')


[OK] manifest ready: benchmarks/single_u15_cross_tgt15.json


In [8]:
# AsymCritic wiring sanity — 确认 train_sac.py 接受 --use-asymmetric-critic 且 SAC agent 走 AsymmetricQNetwork 路径
import subprocess
help_out = subprocess.run(
    ['python', '-m', 'scripts.train_sac', '--help'],
    capture_output=True, text=True, timeout=30,
)
help_text = help_out.stdout + help_out.stderr
assert '--use-asymmetric-critic' in help_text, 'train_sac.py does not expose --use-asymmetric-critic'
print('[OK] --use-asymmetric-critic flag present in train_sac.py CLI')

# 检查 §7.6.4 vanilla baseline 存在（用于事后对比）
vanilla_final = X_VANILLA_BASELINE_ROOT / 'results' / 'final_eval.json'
if vanilla_final.exists():
    bl = json.loads(vanilla_final.read_text(encoding='utf-8'))
    print(f'[OK] §7.6.4 vanilla baseline 已就位: final={bl["eval_success_rate"]:.4f}  counts={bl.get("eval_termination_counts", {})}')
else:
    print(f'[WARN] §7.6.4 vanilla baseline 不存在: {vanilla_final}')
    print('       本 notebook 仍可独立跑，但 fig combined diff 会缺基线列')


[OK] --use-asymmetric-critic flag present in train_sac.py CLI
[OK] §7.6.4 vanilla baseline 已就位: final=0.1000  counts={'out_of_bounds': 20, 'timeout': 7, 'goal': 3}


## 4. Train — single_cross_s0 + AsymCritic (1.0M, skip/resume)


In [ ]:
x_state_path = X_RUN_ROOT / 'trainer_state.json'
if x_state_path.exists():
    x_state = json.loads(x_state_path.read_text(encoding='utf-8'))
    x_current_step = int(x_state.get('env_step', 0))
else:
    x_current_step = 0
print(f'[state] X env_step = {x_current_step:,} / target {X_TOTAL_STEPS:,}')

if x_current_step >= X_TOTAL_STEPS:
    print(f'[skip] X already trained to {x_current_step:,} >= {X_TOTAL_STEPS:,}')
elif x_current_step > 0:
    print(f'[resume] X continuing from {x_current_step:,} -> {X_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --resume {str(X_RUN_ROOT)} \
        --total-steps {X_TOTAL_STEPS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {str(X_MANIFEST_PATH)} \
        --device {DEVICE}
else:
    print(f'[train] X fresh start -> {X_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --flow {X_FLOW_PATH} \
        --task-geometry {X_TASK_GEOMETRY} \
        --target-speed {TARGET_SPEED} \
        --objective {OBJECTIVE} \
        --probe-layout {PROBE_LAYOUT} \
        --history-length {HISTORY_LENGTH} \
        --total-steps {X_TOTAL_STEPS} \
        --random-steps {RANDOM_STEPS} \
        --update-after {UPDATE_AFTER} \
        --batch-size {BATCH_SIZE} \
        --hidden-dim {HIDDEN_DIM} \
        --num-envs {NUM_ENVS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {str(X_MANIFEST_PATH)} \
        --seed {SEED} \
        --device {DEVICE} \
        --save-dir {str(X_RUN_ROOT)} \
        --checkpoint-dir {str(X_CKPT_ROOT)} \
        --use-asymmetric-critic


[state] X env_step = 0 / target 1,000,000
[train] X fresh start -> 1,000,000
[train] episode=5 step=642 return=-251.36 success=False time=17.5s geometry=cross_stream history=4
[train] episode=10 step=1122 return=-414.93 success=False time=93.3s geometry=cross_stream history=4
[train] episode=15 step=1326 return=-520.75 success=False time=110.4s geometry=cross_stream history=4
[train] episode=20 step=1878 return=-403.95 success=False time=61.3s geometry=cross_stream history=4
[train] episode=25 step=2712 return=-340.63 success=False time=68.7s geometry=cross_stream history=4
[train] episode=30 step=3330 return=-411.30 success=False time=38.1s geometry=cross_stream history=4
[train] episode=35 step=3954 return=-267.16 success=False time=8.0s geometry=cross_stream history=4
[train] episode=40 step=4350 return=-264.62 success=False time=11.7s geometry=cross_stream history=4
[train] episode=45 step=5046 return=-429.74 success=False time=104.3s geometry=cross_stream history=4 | q1=511.421 ac

## 5. Summary + gate + ablation diff vs §7.6.4 vanilla baseline


In [ ]:
def summarize_phase(run_root: Path, total_steps: int, label: str, gate_filename: str):
    eval_log_path = run_root / 'results' / 'eval_log.csv'
    final_eval_path = run_root / 'results' / 'final_eval.json'
    trainer_state_path = run_root / 'trainer_state.json'

    if not eval_log_path.exists():
        raise FileNotFoundError(f'missing eval log: {eval_log_path}')
    if not final_eval_path.exists():
        raise FileNotFoundError(f'missing final eval: {final_eval_path}')

    df = pd.read_csv(eval_log_path)
    final_eval = json.loads(final_eval_path.read_text(encoding='utf-8'))
    trainer_state = json.loads(trainer_state_path.read_text(encoding='utf-8')) if trainer_state_path.exists() else {}

    peak_success = float(df['eval_success_rate'].max()) if len(df) else 0.0
    peak_step = int(df.loc[df['eval_success_rate'].idxmax(), 'env_step']) if len(df) else 0
    last100 = df[df['env_step'] >= total_steps - 100_000].copy()
    last100_mean = float(last100['eval_success_rate'].mean()) if len(last100) else 0.0
    final_success = float(final_eval['eval_success_rate'])
    counts = final_eval.get('eval_termination_counts', {})
    num_eps = float(final_eval.get('num_eval_episodes', EVAL_EPISODES))
    oob_rate = float(counts.get('out_of_bounds', 0)) / max(num_eps, 1.0)

    print('=' * 100)
    print(f'{label}  (arrival_v2 / {PROBE_LAYOUT} / sac_asym / {total_steps:,} steps)')
    print('-' * 100)
    print(f"  final_success_rate    : {final_success:.4f}   gate >= {PASS_FINAL_SUCCESS:.2f}")
    print(f"  peak_success_rate     : {peak_success:.4f}   @ {peak_step:,}")
    print(f"  last100k_mean_success : {last100_mean:.4f}   gate >= {PASS_LAST100_RATIO * peak_success:.4f}")
    print(f"  final_oob_rate        : {oob_rate:.4f}   gate <= {PASS_OOB_RATE:.2f}")
    print(f"  obs_dim               : {trainer_state.get('observation_dim', 'NA')}")
    print(f"  context_obs           : {trainer_state.get('include_episode_context_obs', 'NA')}")
    print(f"  timeout_bootstrap     : {trainer_state.get('timeout_bootstrap_semantics', 'NA')}")
    print(f"  asym_critic_enabled   : {trainer_state.get('use_asymmetric_critic', trainer_state.get('privileged_obs_dim', 'NA'))}")
    print(f"  termination           : {counts}")
    print('=' * 100)

    if len(df):
        print()
        print('[last 16 eval rows]')
        cols = ['env_step', 'eval_success_rate', 'eval_return', 'eval_safety_cost',
                'eval_time_s', 'eval_progress_ratio']
        available = [c for c in cols if c in df.columns]
        print(df[available].tail(16).to_string(index=False))

    checks = [
        ('final success >= 0.85', final_success >= PASS_FINAL_SUCCESS, f'{final_success:.4f}'),
        ('last100k mean >= 0.9 * peak', last100_mean >= PASS_LAST100_RATIO * peak_success,
         f'{last100_mean:.4f} / peak={peak_success:.4f}'),
        ('final OOB rate <= 0.10', oob_rate <= PASS_OOB_RATE, f'{oob_rate:.4f}'),
        ('arrival_v2 context obs enabled',
         trainer_state.get('include_episode_context_obs') is True,
         str(trainer_state.get('include_episode_context_obs'))),
        ('arrival_v2 timeout terminal semantics',
         trainer_state.get('timeout_bootstrap_semantics') == 'terminal',
         str(trainer_state.get('timeout_bootstrap_semantics'))),
    ]

    print()
    print('=' * 96)
    print(f"{'check':<48}{'pass':>8}{'detail':>40}")
    print('-' * 96)
    all_pass = True
    for name, ok, detail in checks:
        mark = 'PASS' if ok else 'FAIL'
        if not ok:
            all_pass = False
        print(f'{name:<48}{mark:>8}{detail:>40}')
    print('=' * 96)

    summary = {
        'phase': label,
        'objective': OBJECTIVE,
        'probe_layout': PROBE_LAYOUT,
        'history_length': HISTORY_LENGTH,
        'seed': SEED,
        'total_steps': total_steps,
        'algorithm': 'sac_asym',
        'final_success_rate': final_success,
        'peak_success_rate': peak_success,
        'peak_step': peak_step,
        'last100k_mean_success': last100_mean,
        'final_oob_rate': oob_rate,
        'termination_counts': counts,
        'all_pass': bool(all_pass),
        'checks': [{'name': n, 'ok': bool(ok), 'detail': d} for n, ok, d in checks],
    }
    out_path = run_root / 'results' / gate_filename
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(f'[saved] {out_path}')
    return summary

x_summary = summarize_phase(
    X_RUN_ROOT,
    X_TOTAL_STEPS,
    'SINGLE_CROSS_S0_ASYM_ABLATION',
    'single_cross_s0_asym_ablation_gate_summary.json',
)
X_PASS = x_summary['all_pass']
print()
print(f'X_PASS = {X_PASS}')


SINGLE_CROSS_S0_ASYM_ABLATION  (arrival_v2 / s0 / sac_asym / 1,000,000 steps)
----------------------------------------------------------------------------------------------------
  final_success_rate    : 0.1667   gate >= 0.85
  peak_success_rate     : 0.2667   @ 950,004
  last100k_mean_success : 0.1167   gate >= 0.2400
  final_oob_rate        : 0.6333   gate <= 0.10
  obs_dim               : 48
  context_obs           : True
  timeout_bootstrap     : terminal
  asym_critic_enabled   : NA
  termination           : {'out_of_bounds': 19, 'goal': 5, 'timeout': 6}

[last 16 eval rows]
 env_step  eval_success_rate  eval_return  eval_safety_cost  eval_time_s  eval_progress_ratio
   600000           0.066667  -228.160087         31.248289   112.963333             0.103177
   625002           0.000000  -235.066244         21.446847   101.810000             0.071923
   650004           0.000000  -222.282604         19.006662    93.443333             0.220866
   675000           0.000000  -222.8

## 6. Ablation verdict — sac_asym(s0) vs sac_vanilla(s0) vs sac_vanilla(s1)


In [ ]:
print('=' * 100)
print('SINGLE_CROSS — AsymCritic ablation verdict')
print('-' * 100)

# 当前 ablation 结果
asym_final = float(x_summary['final_success_rate'])
asym_oob = float(x_summary['final_oob_rate'])
asym_peak = float(x_summary['peak_success_rate'])
asym_peak_step = int(x_summary['peak_step'])

# §7.6.4 vanilla baseline (s0_k4) — 实际读盘
vanilla_s0_final_path = X_VANILLA_BASELINE_ROOT / 'results' / 'final_eval.json'
if vanilla_s0_final_path.exists():
    vs0 = json.loads(vanilla_s0_final_path.read_text(encoding='utf-8'))
    vs0_final = float(vs0['eval_success_rate'])
    vs0_counts = vs0.get('eval_termination_counts', {})
    vs0_oob = float(vs0_counts.get('out_of_bounds', 0)) / max(float(vs0.get('num_eval_episodes', EVAL_EPISODES)), 1.0)
else:
    print(f'[WARN] §7.6.4 vanilla baseline 未找到，用 §7.6 report 转载值: final=0.100, oob=0.667')
    vs0_final, vs0_oob = 0.100, 0.667

# §7.1 s1 baseline — 从 report 转载（s1_k4 路径若不存在，则用 report 值）
vs1_final_path = X_S1_BASELINE_ROOT / 'results' / 'final_eval.json'
if vs1_final_path.exists():
    vs1 = json.loads(vs1_final_path.read_text(encoding='utf-8'))
    vs1_final = float(vs1['eval_success_rate'])
    vs1_counts = vs1.get('eval_termination_counts', {})
    vs1_oob = float(vs1_counts.get('out_of_bounds', 0)) / max(float(vs1.get('num_eval_episodes', EVAL_EPISODES)), 1.0)
else:
    print(f'[INFO] §7.1 s1 baseline 用 report 转载值: final=0.900, oob=0.100')
    vs1_final, vs1_oob = 0.900, 0.100

print()
print(f'{"config":<40}{"final":>10}{"oob":>10}{"Δ vs vanilla_s0":>20}{"Δ vs vanilla_s1":>20}')
print('-' * 100)
print(f'{"vanilla SAC, s1_k4 (§7.1 ref)":<40}{vs1_final:>10.4f}{vs1_oob:>10.4f}{"-":>20}{"-":>20}')
print(f'{"vanilla SAC, s0_k4 (§7.6.4 baseline)":<40}{vs0_final:>10.4f}{vs0_oob:>10.4f}{"-":>20}{vs0_final-vs1_final:>+20.4f}')
print(f'{"AsymCritic, s0_k4 (this run)":<40}{asym_final:>10.4f}{asym_oob:>10.4f}{asym_final-vs0_final:>+20.4f}{asym_final-vs1_final:>+20.4f}')
print('=' * 100)

# 判读
gap_closed = asym_final - vs0_final
gap_remaining = vs1_final - asym_final
print()
print(f'AsymCritic 相对 vanilla_s0 的 final_success 增益 : {gap_closed:+.4f}  ({gap_closed * 100:+.1f}pp)')
print(f'相对 vanilla_s1 ref 仍剩 gap                  : {gap_remaining:+.4f}  ({gap_remaining * 100:+.1f}pp)')
print()

if asym_final >= PASS_FINAL_SUCCESS and asym_oob <= PASS_OOB_RATE:
    verdict = 'PASS — AsymCritic 单独闭合 80pp gap，论文可写"privileged hull-integral flow 充分"'
elif asym_final >= 0.40:
    verdict = 'PARTIAL — AsymCritic 有效但未达 gate；下一步走 A (sac_asym_lnutd) 验证组合是否补齐'
elif asym_final > vs0_final + 0.10:
    verdict = 'WEAK — AsymCritic 有 marginal 提升但 dominate failure mode 未改变；critic estimation 误差不是主因'
else:
    verdict = 'NO-EFFECT — AsymCritic 未改善 single_cross_s0 catastrophic failure；下一步重定向到 reward shaping / exploration / curriculum'

print(f'>>> verdict: {verdict}')

# 落盘 ablation summary
ablation_out = {
    'experiment': 'arrival_v2_s0_cross_asym_ablation',
    'seed': SEED,
    'benchmark': X_BENCHMARK_KEY,
    'probe_layout': PROBE_LAYOUT,
    'total_steps': X_TOTAL_STEPS,
    'algorithm': 'sac_asym',
    'cli_diff_vs_vanilla': '--use-asymmetric-critic',
    'results': {
        'sac_asym_s0': {'final': asym_final, 'oob': asym_oob, 'peak': asym_peak, 'peak_step': asym_peak_step},
        'sac_vanilla_s0_baseline': {'final': vs0_final, 'oob': vs0_oob},
        'sac_vanilla_s1_reference': {'final': vs1_final, 'oob': vs1_oob},
    },
    'gap_closed_vs_vanilla_s0_pp': round(gap_closed * 100, 2),
    'gap_remaining_vs_vanilla_s1_pp': round(gap_remaining * 100, 2),
    'all_pass': bool(x_summary['all_pass']),
    'verdict': verdict,
}
out_dir = Path('experiments/arrival_v2_prototype/s0_cross_asym_ablation_summary')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'combined_gate_summary.json'
out_path.write_text(json.dumps(ablation_out, indent=2), encoding='utf-8')
print(f'\n[saved] {out_path}')


SINGLE_CROSS — AsymCritic ablation verdict
----------------------------------------------------------------------------------------------------

config                                       final       oob     Δ vs vanilla_s0     Δ vs vanilla_s1
----------------------------------------------------------------------------------------------------
vanilla SAC, s1_k4 (§7.1 ref)               0.9000    0.1000                   -                   -
vanilla SAC, s0_k4 (§7.6.4 baseline)        0.1000    0.6667                   -             -0.8000
AsymCritic, s0_k4 (this run)                0.1667    0.6333             +0.0667             -0.7333

AsymCritic 相对 vanilla_s0 的 final_success 增益 : +0.0667  (+6.7pp)
相对 vanilla_s1 ref 仍剩 gap                  : +0.7333  (+73.3pp)

>>> verdict: NO-EFFECT — AsymCritic 未改善 single_cross_s0 catastrophic failure；下一步重定向到 reward shaping / exploration / curriculum

[saved] experiments/arrival_v2_prototype/s0_cross_asym_ablation_summary/combined_gate_summary